In [2]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, VBox, interactive_output
from IPython.display import display

# Base parameters for the original signal
N_base = 16
m_freq = 3  # Fixed frequency index matching the textbook example

def plot_zero_padding_demo_with_captions(L_total):
    plt.close('all')
    
    # 1. Time domain setup
    n_base = np.arange(N_base)
    x_base = np.cos(2 * np.pi * m_freq * n_base / N_base)
    
    # Zero-padded signal of total length L_total
    x_padded = np.zeros(L_total)
    x_padded[:N_base] = x_base
    n_full = np.arange(L_total)
    
    # 2. Frequency domain setup (DFT of length L_total)
    X_dft = np.fft.fft(x_padded)
    freqs = np.arange(L_total) * (N_base / L_total)
    
    # High-resolution continuous spectrum (DTFT) for smooth red curve
    n_fft = 4096
    x_dtft_buf = np.zeros(n_fft)
    x_dtft_buf[:N_base] = x_base
    X_dtft = np.fft.fft(x_dtft_buf, n_fft)
    freqs_dtft = np.fft.fftfreq(n_fft, d=1/N_base)
    
    # 3. Plotting layout: 2 subplots (Time Domain left, Frequency Domain right)
    fig, (ax_t, ax_f) = plt.subplots(1, 2, figsize=(15, 5.5))
    
    # --- TIME DOMAIN PLOT ---
    ax_t.plot(n_full, x_padded, 'r-', lw=1.2)
    ax_t.stem(n_full, x_padded, linefmt='C0-', markerfmt='C0o', basefmt='k-')
    ax_t.set_title(f"Amplitude of Input Signal (N = {L_total} samples)", fontsize=11, fontweight='bold')
    ax_t.set_xlabel("Time ($n$)", fontsize=10)
    ax_t.set_ylabel("Amplitude", fontsize=10)
    ax_t.set_xlim(-1, L_total)
    ax_t.set_ylim(-1.2, 1.2)
    ax_t.grid(True, linestyle='--', alpha=0.6)
    
    # Text box / caption below Time Domain plot
    time_caption = (
        f"• Active samples: {N_base} (non-zero part)\n"
        f"• Zero-padded samples: {L_total - N_base} zeros added at the end\n"
        f"• Total sequence length: L = {L_total}"
    )
    ax_t.text(0.5, -0.28, time_caption, transform=ax_t.transAxes, fontsize=9,
              verticalalignment='top', horizontalalignment='center',
              bbox=dict(boxstyle='round,pad=0.5', facecolor='#f8f9fa', edgecolor='#ced4da'))
    
    # --- FREQUENCY DOMAIN PLOT ---
    pos_mask = (freqs_dtft >= 0) & (freqs_dtft <= N_base / 2)
    ax_f.plot(freqs_dtft[pos_mask], np.abs(X_dtft[pos_mask]) * (N_base / L_total), 'r-', lw=1.5, label='Continuous Spectrum')
    
    dft_mask = freqs <= N_base / 2
    ax_f.stem(freqs[dft_mask], np.abs(X_dft[dft_mask]) * (N_base / L_total), 
              linefmt='C0-', markerfmt='C0o', basefmt='k-', label=f'DFT Points (L={L_total})')
              
    ax_f.set_title(f"Magnitude of DFT Transform (Points = {L_total})", fontsize=11, fontweight='bold')
    ax_f.set_xlabel("Freq Index", fontsize=10)
    ax_f.set_ylabel("Magnitude", fontsize=10)
    ax_f.set_xlim(0, N_base / 2)
    ax_f.grid(True, linestyle='--', alpha=0.6)
    ax_f.legend(loc='upper right', fontsize=8)
    
    # Text box / caption below Frequency Domain plot
    freq_caption = (
        f"• Number of DFT frequency points: {L_total}\n"
        f"• Observations: Points densify and trace the continuous DTFT curve.\n"
        f"• Note: Resolution (main lobe width) remains unchanged."
    )
    ax_f.text(0.5, -0.28, freq_caption, transform=ax_f.transAxes, fontsize=9,
              verticalalignment='top', horizontalalignment='center',
              bbox=dict(boxstyle='round,pad=0.5', facecolor='#f8f9fa', edgecolor='#ced4da'))
    
    # Adjust layout to make room for captions
    plt.subplots_adjust(bottom=0.22)
    plt.show()

# Slider allowing arbitrary lengths (not strictly restricted to powers of 2, step=1)
len_slider = IntSlider(value=16, min=16, max=128, step=1, description='Total Length (N):', style={'description_width': '120px'})
display(VBox([len_slider, interactive_output(plot_zero_padding_demo_with_captions, {'L_total': len_slider})]))